# Day 2 · §2.8.4 — What a Neural Network Actually Is  *(Instructor Demo)*

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

> **Instructor-led demo, not a graded student lab.** Runs on Google Colab (which has TensorFlow
> pre-installed). A tiny neural network on the **same auth-log features from Lab 3c** — so we can
> compare apples to apples.

### The plain-English idea (no calculus)
- A **neuron** is just a *weighted sum + a squish*: multiply each input by a learned *weight*, add
  them up, and squish the total into a useful range.
- A **network** stacks neurons in layers: input → hidden layer(s) → output.
- **"A trained network"** is nothing mystical — it's just the **learned weights**.
- **Training loop:** predict → measure how wrong (*loss / error*) → nudge the weights to be less
  wrong (*gradient descent*) → repeat.
- **Epoch:** one **epoch = one full pass through all the training data.** We train for many epochs;
  after each one the network's error should be a little lower. Watching that error fall *is* watching
  the network learn.
- **Overfitting:** train too long and the net *memorizes* the training data — its error on the
  training set keeps falling while its error on unseen data starts to **rise**. We'll see that live.


## Step 1 — Tools (TensorFlow/Keras is built into Colab)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
print("TensorFlow version:", tf.__version__)

## Step 2 — Rebuild the exact Lab 3c features

In [ ]:
import os
def load_logs(fname="day2_auth_logs.csv", url=""):
    if url: return pd.read_csv(url, parse_dates=["Login Timestamp"])
    for p in [fname, "data/"+fname]:
        if os.path.exists(p): return pd.read_csv(p, parse_dates=["Login Timestamp"])
    from google.colab import files
    up=files.upload(); return pd.read_csv(list(up.keys())[0], parse_dates=["Login Timestamp"])

df = load_logs()
ua = df["User Agent String"].fillna("").str.lower()
fails_per_ip = df.loc[df["Login Successful"]==False].groupby("IP Address").size()
X = pd.DataFrame({
    "login_successful": df["Login Successful"].astype(int),
    "is_scripted_ua":   ua.str.contains("python-requests|curl|go-http|wget|scrapy|okhttp").astype(int),
    "is_foreign":       (df["Country"]!="NO").astype(int),
    "hour":             df["Login Timestamp"].dt.hour,
    "ip_fail_count":    df["IP Address"].map(fails_per_ip).fillna(0).astype(int),
})
y = df["Is Attack IP"].astype(int)

# Split, then scale on training only (neural nets REALLY need scaled inputs - Lab 3b).
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = MinMaxScaler().fit(X_tr); X_tr_s = sc.transform(X_tr); X_te_s = sc.transform(X_te)
print("Ready:", X_tr_s.shape[0], "train /", X_te_s.shape[0], "test rows, ", X_tr_s.shape[1], "features.")

## Step 3 — Build a tiny neural network

Two small hidden layers of neurons, then one output neuron giving the probability of "attack".
`relu` is the squish for hidden layers; `sigmoid` squishes the output to 0-1.


In [ ]:
model = Sequential([
    Dense(8, activation="relu", input_shape=(X_tr_s.shape[1],)),  # hidden layer: 8 neurons
    Dense(8, activation="relu"),                                   # hidden layer: 8 neurons
    Dense(1, activation="sigmoid"),                               # output: probability of attack
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## Step 4 — Train it: many epochs, and keep the history

Remember: **one epoch = one full pass through all the training data.** We ask for 40 epochs and
record the error (**loss**) and accuracy after every single one, for both the training set and the
held-out validation set.


In [ ]:
history = model.fit(
    X_tr_s, y_tr,
    validation_data=(X_te_s, y_te),
    epochs=40, batch_size=64, verbose=0)   # verbose=0 keeps the output tidy; history holds the per-epoch numbers
print("Trained for", len(history.history["loss"]), "epochs.")

## Step 5 — Watch the ERROR shrink, epoch by epoch  ⭐

This is the teaching moment. `history.history["loss"]` is a list with **one error value per epoch**.
Early on the network is basically guessing (high error); each epoch it nudges its weights and the
error drops. Let's print a few checkpoints, then plot the whole curve.


In [ ]:
loss = history.history["loss"]          # training error after each epoch
val_loss = history.history["val_loss"]  # error on unseen validation data after each epoch

# Print the error at a few epochs so students SEE it fall.
print("epoch |  training error (loss)")
print("------+------------------------")
for e in [0, 4, 9, 19, 29, 39]:
    if e < len(loss):
        print(f"  {e+1:3d} |  {loss[e]:.3f}")
print(f"\nError fell from {loss[0]:.3f} (epoch 1) to {loss[-1]:.3f} (epoch {len(loss)}) — the network learned.")

In [ ]:
# Plot the error curve. Training error keeps falling; validation error falls, then may creep back up.
plt.figure(figsize=(7,4))
plt.plot(loss, label="training error", color="#1E5199")
plt.plot(val_loss, label="validation error (unseen data)", color="#C0392B")
best = int(np.argmin(val_loss))
plt.axvline(best, ls="--", color="gray")
plt.text(best+0.3, max(val_loss)*0.9, "  lowest validation\n  error = stop here", color="gray")
plt.xlabel("epoch (one full pass through the data)"); plt.ylabel("error (loss) — lower is better")
plt.title("Watch it learn: error shrinks each epoch"); plt.legend(); plt.tight_layout(); plt.show()

**Say it out loud:** every epoch, the network made a prediction, measured how wrong it was, and
nudged its weights to be a little less wrong. That falling blue line *is* learning. Notice the red
line (error on data it never trained on) stops improving and starts to rise — that's the first sign
of **overfitting**, which the next step shows from the accuracy side.


## Step 6 — The same story in accuracy (the overfitting view)

In [ ]:
h = history.history
plt.figure(figsize=(7,4))
plt.plot(h["accuracy"],     label="training accuracy",   color="#1E5199")
plt.plot(h["val_accuracy"], label="validation accuracy", color="#C0392B")
best = int(np.argmax(h["val_accuracy"]))
plt.axvline(best, ls="--", color="gray")
plt.xlabel("epoch"); plt.ylabel("accuracy")
plt.title("Overfitting: training keeps rising, validation turns down")
plt.legend(); plt.tight_layout(); plt.show()

**Discuss:** after the dashed line the network is *memorizing* the training rows — training
accuracy climbs but it gets **worse** on unseen data. Fixes: *early stopping* (stop at the best
validation epoch), more data, or `Dropout`. This is the "too-good-is-dangerous" warning from
Day 1 §1.2, made concrete.


## Step 7 — Compare the network to the Lab 3c Decision Tree

In [ ]:
proba = model.predict(X_te_s, verbose=0).ravel()   # attack probability for each test row
nn_pred = (proba >= 0.5).astype(int)               # turn probability into yes/no at 0.5

cm = confusion_matrix(y_te, nn_pred, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=["benign","attack"]).plot(cmap="Blues", colorbar=False)
plt.title("Neural network - test-set confusion matrix"); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"Neural net -> TP={tp} FN={fn} FP={fp} TN={tn}")

**Talking points:**
- On this small, mostly-tabular problem the neural net is **not** magically better than the Lab 3c
  Decision Tree — with few features, the glass-box tree is competitive *and* readable.
- Neural nets earn their keep on **big, complex** data (images, text) — which is why the next demo
  (§2.8.5) moves to **images** with a CNN.
- Trade-off preview for §2.10: the tree tells you *why*; this network is a **black box** — you'd need
  an explainer (LIME/SHAP) to justify its alerts.


## Wrap-up
- A neuron = weighted sum + activation; a trained network = the learned weights.
- **One epoch = one full pass through the training data;** the error falls a little each epoch —
  that falling curve is the network learning.
- We watched **overfitting** live (validation error/accuracy turning the wrong way) and named the fixes.
- More power isn't always better: match the model to the data. Next we go where neural nets shine —
  images — with a **CNN**.
